In [ ]:
!pip install -U transformers
!pip install datasets

In [ ]:
import torch
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup, \
    ModernBertForMultipleChoice
from datasets import load_dataset

from tqdm import tqdm
from collections import Counter

from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

print(f'on device: {device}')

## domain adapted or off-the-shelf model

# model_id = 'FacebookAI/roberta-large'
# model_id = 'answerdotai/ModernBERT-large'
# model_id = 'google/bigbird-roberta-large'
# model_id = '../../models/bidirectional_attn/roberta-large/cleaned/last_checkpoint'
model_id = 'allenai/longformer-large-4096'

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)


class HFDataset(Dataset):

    def __init__(self, hf):
        self.ds = list(hf.map(HFDataset.process_fn, batched=True, batch_size=8, remove_columns=hf.column_names))

    @staticmethod
    def process_fn(b):
        question, opa, opb, opc, opd, cop = b['question'], b['opa'], b['opb'], b['opc'], b['opd'], b['cop']

        return {
            'question': question,
            'options': list(zip(opa, opb, opc, opd)),
            'answer': cop
        }

    def __getitem__(self, index):
        return self.ds[index]

    def __len__(self):
        return len(self.ds)


class MedMCQA(Dataset):

    def __init__(self, dataset_id='openlifescienceai/medmcqa'):
        train_data, test_data = load_dataset(dataset_id, split=['train', 'validation'])
        self.train = HFDataset(train_data.select(range(300))).ds
        self.test = HFDataset(test_data).ds

    def __getitem__(self, index):
        raise NotImplementedError('Object of type MedMCQA has no index. Use ".train", ".val", or ".test" instead.')

    def __len__(self):
        raise NotImplementedError('Object of type MedMCQA has no len')

    @staticmethod
    def collate_fn(batch):
        qs, ops, ans = [], [], []

        for e in batch:
            qs.extend([e['question']] * 4)
            ops.extend(e['options'])
            ans.append(e['answer'])

        # dynamically truncate to max_model_seq_length and pad to batch_max_seq
        inputs = tokenizer(qs, ops, return_tensors='pt', padding=True, truncation=True).to(device)
        labels = torch.tensor(ans, dtype=torch.long).to(device)

        batch_size = len(batch)
        num_choices = 4
        seq_len = inputs['input_ids'].shape[1]

        return {k: v.view(batch_size, num_choices, seq_len).to(device) for k, v in inputs.items()}, labels


medmcqa = MedMCQA()

train_loader = DataLoader(medmcqa.train, batch_size=8, shuffle=True, collate_fn=MedMCQA.collate_fn)
test_loader = DataLoader(medmcqa.test, batch_size=8, shuffle=True, collate_fn=MedMCQA.collate_fn)

if 'ModernBERT' in model_id:
    model = ModernBertForMultipleChoice.from_pretrained(model_id).to(device)
else:
    model = AutoModelForMultipleChoice.from_pretrained(model_id).to(device)

params = sum(p.numel() for p in model.parameters())
print(f'params: {round(params / (10 ** 9), 4)} B')

## trainable parameters
# params2train = [n for n, p in model.named_parameters() if p.requires_grad]

if 'ModernBERT' in model_id:
    backbone = model.model.parameters()
elif 'bigbird' in model_id:
    backbone = model.bert.parameters()
elif 'longformer' in model_id:
    backbone = model.longformer.parameters()
else:
    backbone = model.roberta.parameters()

optimizer = AdamW([
    {"params": backbone, "lr": 2e-5},
    {"params": model.classifier.parameters(), "lr": 1e-4},
])

epochs = 10
num_training_steps = epochs * len(train_loader)

num_warmup_steps = int(0.1 * num_training_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

## train
for epoch in tqdm(range(epochs), desc='going in epoch'):

    loss_epoch = 0.0

    print('training instantiated')

    model.train()

    for inputs, labels in train_loader:
        # forward pass
        ## inputs.input_ids = (batch_size, num_choices, seq_length)
        ## labels = (batch_size, )

        optimizer.zero_grad(set_to_none=True)
        J = model(**inputs, labels=labels)

        logits, loss = J.logits, J.loss
        loss_epoch += loss.item()

        # backward pass
        loss.backward()
        clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

    print(f'\n\nepoch: {epoch + 1} loss: {loss_epoch / len(train_loader)}\n\n')
    print('running per epoch evaluation on test data')

    model.eval()
    with torch.no_grad():
        Y_pred, Y_gold = [], []
        for inputs, labels in train_loader:
            out = model(**inputs)
            pred = out.logits.argmax(dim=1)

            Y_pred.extend(pred.cpu().tolist())
            Y_gold.extend(labels.cpu().tolist())

    print("\nPred dist:", Counter(Y_pred))
    print("Gold dist:", Counter(Y_gold))

    p_macro = precision_score(Y_gold, Y_pred, average='macro')
    r_macro = recall_score(Y_gold, Y_pred, average='macro')
    f1_macro = f1_score(Y_gold, Y_pred, average='macro')

    ## same as f1-score_micro
    acc = accuracy_score(Y_gold, Y_pred)

    evald = {'accuracy': acc, 'P-macro': p_macro, 'R-macro': r_macro, 'F1-macro': f1_macro}

    print(*evald.items(), sep='\n')

torch.cuda.empty_cache()

print('After training inference')

model.eval()
with torch.no_grad():
    Y_pred, Y_gold = [], []
    for inputs, labels in train_loader:
        out = model(**inputs)
        pred = out.logits.argmax(dim=1)

        Y_pred.extend(pred.cpu().tolist())
        Y_gold.extend(labels.cpu().tolist())

    print("\nPred dist:", Counter(Y_pred))
    print("Gold dist:", Counter(Y_gold))

    p_macro = precision_score(Y_gold, Y_pred, average='macro')
    r_macro = recall_score(Y_gold, Y_pred, average='macro')
    f1_macro = f1_score(Y_gold, Y_pred, average='macro')

    ## same as f1-score_micro
    acc = accuracy_score(Y_gold, Y_pred)

    evald = {'accuracy': acc, 'P-macro': p_macro, 'R-macro': r_macro, 'F1-macro': f1_macro}

    print(*evald.items(), sep='\n')